# OpenPlaque — Left-Coronary Backbone Branch Discovery v1.2
C7-control robustness correction: source-aligned vesselness, safe failed paths, and multi-start branch-origin seeding. Scientific acceptance gates remain unchanged. Run with **Runtime → Run all**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, sys
os.chdir('/content')
print('Working directory repaired:', os.getcwd())
DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
OUTPUT = DRIVE_ROOT / 'Left_Coronary_Backbone_Branch_Discovery_v1'
REUSE_EXISTING_OUTPUT = False
if OUTPUT.exists() and not REUSE_EXISTING_OUTPUT:
    shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'notebook_started.json').write_text(json.dumps({'status':'STARTED','notebook':'v1.2-control-multistart'}, indent=2))
print('Output:', OUTPUT)

In [ ]:
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
BRANCH = 'left-coronary-backbone-branch-discovery-from-main'
PINNED_SCIENCE_COMMIT = '678ea3063ae25cdc28919fb7ca97593400adc4c2'
repo = '/content/OpenPlaque_backbone_branch_v12'
os.chdir('/content')
if os.path.exists(repo):
    shutil.rmtree(repo)
!git clone --depth 40 --branch "$BRANCH" https://github.com/pazzani/OpenPlaque.git "$repo"
!git -C "$repo" checkout --detach "$PINNED_SCIENCE_COMMIT"
HEAD_OUT = get_ipython().getoutput(f'git -C {repo} rev-parse HEAD')
MB_OUT = get_ipython().getoutput(f'git -C {repo} merge-base HEAD {BASELINE}')
HEAD = HEAD_OUT[-1].strip() if HEAD_OUT else ''
MB = MB_OUT[-1].strip() if MB_OUT else ''
print('Checked out:', HEAD)
print('Merge base:', MB)
assert HEAD == PINNED_SCIENCE_COMMIT, f'checkout failed: {HEAD_OUT}'
assert MB == BASELINE, f'merge-base check failed: {MB_OUT}'
%pip install -q /content/OpenPlaque_backbone_branch_v12
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
os.chdir(repo)
print('Repository cwd:', os.getcwd())

In [ ]:
import numpy as np, pytest
from openplaque.left_coronary_backbone_branch_discovery_v1_1 import synthetic_local_vesselness_self_test, synthetic_short_path_self_test
from openplaque.left_coronary_backbone_branch_discovery_v1_2 import synthetic_multistart_seed_self_test
print('Synthetic local-vesselness self-test:', synthetic_local_vesselness_self_test())
print('Synthetic short-path self-test:', synthetic_short_path_self_test())
print('Synthetic multistart-seed self-test:', synthetic_multistart_seed_self_test())
rc = pytest.main(['-q', 'tests/test_left_coronary_backbone_branch_discovery_v1.py', 'tests/test_left_coronary_backbone_branch_discovery_v1_1.py', 'tests/test_left_coronary_backbone_branch_discovery_v1_2.py'])
assert rc == 0, f'pytest failed with code {rc}'

In [ ]:
src_path = DRIVE_ROOT / 'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy'
old_ves_path = DRIVE_ROOT / 'Cache/Secondary_3D_Vesselness_Topology_v1/vesselness.npy'
src = np.load(src_path, mmap_mode='r')
old_ves = np.load(old_ves_path, mmap_mode='r')
preflight = {
    'status':'COMPLETE',
    'science_commit': PINNED_SCIENCE_COMMIT,
    'source_shape': list(src.shape),
    'legacy_cached_vesselness_shape': list(old_ves.shape),
    'vesselness_fix':'local full-resolution source-space Frangi ROI',
    'short_path_fix':'one-point failed beams recorded without numpy.gradient',
    'control_seed_fix':'arc +/-0.4 mm plus 0.8-mm cross-sectional seed ring; six beam traces retained globally'
}
(OUTPUT / 'preflight_complete.json').write_text(json.dumps(preflight, indent=2))
del src, old_ves
print(json.dumps(preflight, indent=2))

In [ ]:
from openplaque.left_coronary_backbone_branch_discovery_v1_2 import run
result = run(drive_root=str(DRIVE_ROOT), output_dir=str(OUTPUT))
print(json.dumps(result['summary'], indent=2, default=str))
print('Report:', result['report'])
print('ZIP:', result['zip'])